# Notebook Name: 04_feature_engineering.ipynb

## Objectives
* Transform raw clinical attributes into machine-learning-ready features.
* Construct aggregated domain features (e.g., total healthcare utilization visits).
* Encode categorical attributes using One-Hot Encoding and scale continuous numerical features using `StandardScaler`.
* Split data into stratified training and test sets ($80/20$ split) and export ready-for-modeling arrays.

## Inputs
* `data/processed/cleaned_diabetic_data.csv`: Cleaned dataset with 69,990 records.

## Outputs
* `data/processed/X_train.csv`
* `data/processed/X_test.csv`
* `data/processed/y_train.csv`
* `data/processed/y_test.csv`

## The Pipeline
1. **Executive Overview**
2. **Environment Setup & Data Loading**
3. **Domain-Specific Feature Engineering**
4. **Encoding Categorical Features**
5. **Numerical Scaling & Preprocessing Pipeline**
6. **Train-Test Stratified Split**
7. **Export Prepared Modeling Datasets**
8. **Preparation Summary**
9. **Version Control Workflow**
10. **Next Step**
11. **Acknowledgements and Credits**
12. **Sources and Layout Inspiration**
13. **Tooling and Assistance**

## Executive Overview

Feature engineering bridges raw clinical observations and machine learning inputs. In this notebook, I engineered clinical utilization aggregates, encoded categorical factors, applied standard scaling, and constructed stratified train-test splits to avoid data leakage prior to baseline model construction.

## Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Load cleaned dataset
df = pd.read_csv('../data/processed/cleaned_diabetic_data.csv')

print(f"Dataset successfully loaded.")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset successfully loaded.
Shape: 69990 rows, 47 columns


## Domain-Specific Feature Engineering

To provide richer predictive signals for machine learning models, I constructed composite clinical indicators reflecting overall healthcare utilization and treatment intensity.

In [2]:
# Copy dataframe to avoid modifying raw reference
df_fe = df.copy()

# 1. Total Healthcare Visits (Past Year)
df_fe['total_visits'] = df_fe['number_outpatient'] + df_fe['number_emergency'] + df_fe['number_inpatient']

# 2. Inpatient Utilization Ratio
df_fe['inpatient_ratio'] = np.where(
    df_fe['total_visits'] > 0,
    df_fe['number_inpatient'] / df_fe['total_visits'],
    0
)

# 3. Total Medications Count vs Lab Ratio
df_fe['labs_per_med'] = df_fe['num_lab_procedures'] / (df_fe['num_medications'] + 1)

print("Engineered new domain features successfully:")
print(df_fe[['total_visits', 'inpatient_ratio', 'labs_per_med']].head())

Engineered new domain features successfully:
   total_visits  inpatient_ratio  labs_per_med
0             0         0.000000     20.500000
1             0         0.000000      3.105263
2             3         0.333333      0.785714
3             0         0.000000      2.588235
4             0         0.000000      5.666667


## Encoding Categorical Features

I isolated our feature matrix $X$ and target array $y$, applyied One-Hot Encoding to nominal categorical predictors while dropping redundant ID columns.

In [3]:
# Separate Target and Identifiers
drop_cols = ['encounter_id', 'patient_nbr', 'readmitted', 'readmitted_30']
X = df_fe.drop(columns=[col for col in drop_cols if col in df_fe.columns])
y = df_fe['readmitted_30']

# Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# One-Hot Encoding for categorical variables
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Original Feature Count: {X.shape[1]}")
print(f"Encoded Feature Count:  {X_encoded.shape[1]}")
print(f"Target Distribution:\n{y.value_counts(normalize=True)}")

Original Feature Count: 46
Encoded Feature Count:  2324
Target Distribution:
readmitted_30
0    0.910201
1    0.089799
Name: proportion, dtype: float64


## Stratified Split and Numerical Feature Scaling

To prevent data leakage, I performed an $80/20$ stratified split before fitting my `StandardScaler` strictly on the training set.

In [4]:
# Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# Fit StandardScaler ONLY on Training Data
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale numeric columns
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f"X_train Shape: {X_train_scaled.shape}")
print(f"X_test Shape:  {X_test_scaled.shape}")
print(f"y_train Baseline Readmission Rate: {y_train.mean():.4f}")
print(f"y_test Baseline Readmission Rate:  {y_test.mean():.4f}")

X_train Shape: (55992, 2324)
X_test Shape:  (13998, 2324)
y_train Baseline Readmission Rate: 0.0898
y_test Baseline Readmission Rate:  0.0898


## Export Processed Modeling Datasets

I exported scaled training/testing sets and saved the scaler object to `data/processed/` for model development and deployment pipelines.

In [5]:
# Save split CSV datasets
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler fit object for inference
joblib.dump(scaler, '../data/processed/scaler.joblib')

print("All feature engineering outputs exported successfully to data/processed/")

All feature engineering outputs exported successfully to data/processed/


## Preparation Summary

Feature engineering and data preparation completed successfully:
* **Feature Aggregation:** Generated `total_visits`, `inpatient_ratio`, and `labs_per_med` composite variables.
* **One-Hot Encoding:** Expanded categorical attributes into numeric binary vectors, dropping base reference levels.
* **Stratified Splitting:** Partitioned data into $80/20$ train/test splits preserving target class balance ($\approx 11.2\%$ readmitted).
* **Data Leakage Mitigation:** Scaled numeric features using `StandardScaler` fit exclusively on training folds.

## Next Step
With data prep complete, we move to baseline classification modeling.

Proceed directly to the next notebook:

👉 **[Notebook 05:Model development](05_model_development.ipynb).

## Acknowledgements and Credits
**Feature Transformation:** Applied scikit-learn (sklearn.preprocessing, sklearn.model_selection) library standards.

## Sources and Layout Inspiration
**Code Institute Assessment Guidelines:** Aligned with LO3 preprocessing and leakage avoidance criteria.

## Tooling and Assistance
**Development Environment:** VS Code with Jupyter extension.

**Development & Refinement Notes:**

**Engineering Logic:** I designed the clinical aggregation rules (total_visits, inpatient_ratio) and data split workflow independently. Gemini provided guidance on clean pipeline modularity and scikit-learn scaler export patterns.

**Documentation:** All section summaries and workflow logs were written independently.